# Module 5: Data Visualization for Machine Learning

**Topics Covered:**
- Matplotlib fundamentals (figures, axes, subplots)
- Seaborn for statistical visualization
- Essential ML plots:
  - Distribution plots (histograms, KDE, boxplots)
  - Correlation heatmaps
  - Feature importance bar charts
  - Learning curves (train/val loss)
  - Confusion matrix heatmap
  - ROC curve
  - Scatter plots (decision boundaries)

> Visualization is how you understand your data, debug your model, and communicate results.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Style settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
np.random.seed(42)

print(f"Matplotlib: {plt.matplotlib.__version__}")
print(f"Seaborn: {sns.__version__}")

---
## 1. Matplotlib Fundamentals

In [ ]:
# The anatomy of a matplotlib figure
fig, ax = plt.subplots(figsize=(8, 4))

x = np.linspace(0, 10, 100)
ax.plot(x, np.sin(x), label='sin(x)', linewidth=2, color='steelblue')
ax.plot(x, np.cos(x), label='cos(x)', linewidth=2, color='coral', linestyle='--')

ax.set_title('Sine and Cosine Functions', fontsize=14, fontweight='bold')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend()
ax.set_xlim(0, 10)
ax.set_ylim(-1.5, 1.5)
plt.tight_layout()
plt.show()

In [ ]:
# Subplots — show multiple plots together (model comparison)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Scatter plot
n = 100
X = np.random.randn(n, 2)
y = (X[:, 0] + X[:, 1] > 0).astype(int)
axes[0].scatter(X[y==0, 0], X[y==0, 1], alpha=0.7, label='Class 0', c='steelblue')
axes[0].scatter(X[y==1, 0], X[y==1, 1], alpha=0.7, label='Class 1', c='coral')
axes[0].set_title('Binary Classification Data')
axes[0].legend()

# Histogram
data = np.random.randn(500)
axes[1].hist(data, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('Feature Distribution')
axes[1].set_xlabel('Value')

# Line plot (learning curve)
epochs = np.arange(1, 51)
train_loss = 1.5 * np.exp(-0.05 * epochs) + np.random.randn(50) * 0.02
val_loss = 1.5 * np.exp(-0.04 * epochs) + np.random.randn(50) * 0.03 + 0.1
axes[2].plot(epochs, train_loss, label='Train', color='steelblue')
axes[2].plot(epochs, val_loss, label='Validation', color='coral')
axes[2].set_title('Learning Curve')
axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 2. Distribution Analysis — Understanding Your Data

In [ ]:
# Generate a realistic ML dataset
np.random.seed(42)
n = 300
df = pd.DataFrame({
    'age':           np.random.normal(35, 12, n).clip(18, 75).astype(int),
    'income':        np.random.lognormal(10.8, 0.5, n).astype(int),
    'credit_score':  np.random.normal(680, 80, n).clip(400, 850).astype(int),
    'years_employed':np.random.exponential(5, n).clip(0, 40),
    'education':     np.random.choice(['HS', 'Bachelor', 'Master', 'PhD'], n, p=[0.3, 0.4, 0.2, 0.1]),
    'default':       np.random.choice([0, 1], n, p=[0.75, 0.25])
})

In [ ]:
# Distribution plots
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Histogram with KDE
sns.histplot(df['age'], bins=20, kde=True, ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Age Distribution')

# Log-normal income
sns.histplot(df['income'], bins=30, kde=True, ax=axes[0,1], color='coral')
axes[0,1].set_title('Income Distribution')

# Credit score by default status
sns.kdeplot(data=df, x='credit_score', hue='default', ax=axes[1,0], fill=True, alpha=0.4)
axes[1,0].set_title('Credit Score by Default Status')

# Box plot
sns.boxplot(data=df, x='education', y='income', ax=axes[1,1],
            order=['HS', 'Bachelor', 'Master', 'PhD'])
axes[1,1].set_title('Income by Education')
axes[1,1].tick_params(axis='x', rotation=30)

plt.suptitle('Exploratory Data Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Exercise 5.1 — Distribution Plots
Create a 2×2 subplot grid showing:
1. Histogram of `years_employed` (use 25 bins, add KDE)
2. Box plot of `credit_score` by `default` status
3. Violin plot of `age` by `education` level
4. Count plot of `education` colored by `default` status

Add proper titles and labels to each subplot.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
# YOUR CODE HERE


In [ ]:
# SOLUTION
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1. Histogram + KDE
sns.histplot(df['years_employed'], bins=25, kde=True, ax=axes[0,0], color='teal')
axes[0,0].set_title('Years Employed Distribution')
axes[0,0].set_xlabel('Years')

# 2. Box plot
sns.boxplot(data=df, x='default', y='credit_score', ax=axes[0,1])
axes[0,1].set_title('Credit Score by Default Status')
axes[0,1].set_xticklabels(['No Default', 'Default'])

# 3. Violin plot
sns.violinplot(data=df, x='education', y='age', ax=axes[1,0],
               order=['HS', 'Bachelor', 'Master', 'PhD'])
axes[1,0].set_title('Age Distribution by Education')

# 4. Count plot
sns.countplot(data=df, x='education', hue='default',
              order=['HS', 'Bachelor', 'Master', 'PhD'], ax=axes[1,1])
axes[1,1].set_title('Education Count by Default Status')
axes[1,1].legend(title='Default', labels=['No', 'Yes'])

plt.tight_layout()
plt.show()

---
## 3. Correlation Heatmap

In [ ]:
# Correlation matrix — find feature relationships
numeric_cols = ['age', 'income', 'credit_score', 'years_employed', 'default']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr), k=1)  # upper triangle mask
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, mask=mask, ax=ax,
    linewidths=0.5, cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Model Evaluation Plots

In [ ]:
# Learning curves — diagnose overfitting/underfitting
np.random.seed(42)
epochs = np.arange(1, 101)

# Simulate overfitting scenario
train_loss = 2.0 * np.exp(-0.08 * epochs) + 0.05 + np.random.randn(100) * 0.01
val_loss   = 2.0 * np.exp(-0.05 * epochs) + 0.25 + np.random.randn(100) * 0.02
train_acc  = 1 - train_loss / 2.5
val_acc    = 1 - val_loss / 2.5

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
ax1.plot(epochs, train_loss, label='Train Loss', color='steelblue', linewidth=2)
ax1.plot(epochs, val_loss, label='Val Loss', color='coral', linewidth=2)
ax1.axvline(x=40, color='gray', linestyle='--', alpha=0.7, label='Early stop')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()

# Accuracy curves
ax2.plot(epochs, train_acc.clip(0, 1), label='Train Acc', color='steelblue', linewidth=2)
ax2.plot(epochs, val_acc.clip(0, 1), label='Val Acc', color='coral', linewidth=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Training & Validation Accuracy')
ax2.legend()

plt.suptitle('Learning Curves — Overfitting Example', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix heatmap
from sklearn.metrics import confusion_matrix

np.random.seed(42)
y_true = np.random.choice([0, 1, 2], 200, p=[0.5, 0.3, 0.2])
y_pred = y_true.copy()
# Add some errors
error_mask = np.random.rand(200) < 0.15
y_pred[error_mask] = np.random.choice([0, 1, 2], error_mask.sum())

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, data, fmt, title in zip(axes, [cm, cm_norm], ['d', '.2f'],
                                  ['Confusion Matrix (counts)', 'Confusion Matrix (normalized)']):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues', ax=ax,
                xticklabels=['Cat', 'Dog', 'Bird'],
                yticklabels=['Cat', 'Dog', 'Bird'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

np.random.seed(42)
n = 500
X_data = np.random.randn(n, 5)
y_data = (X_data[:, 0] + 2*X_data[:, 1] - X_data[:, 2] + np.random.randn(n) > 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.3, random_state=42)
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)
y_scores = model.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_scores)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC = 0.5)')
ax.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance bar chart
from sklearn.ensemble import RandomForestClassifier

feature_names = ['Credit Score', 'Income', 'Age', 'Debt Ratio', 'Years Employed']
rf = RandomForestClassifier(n_estimators=50, random_state=42)
rf.fit(X_train, y_train)
importances = rf.feature_importances_

sorted_idx = np.argsort(importances)
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(np.array(feature_names)[sorted_idx], importances[sorted_idx],
               color=plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(feature_names))))
ax.set_xlabel('Importance Score')
ax.set_title('Feature Importances (Random Forest)', fontweight='bold')
for bar, imp in zip(bars, importances[sorted_idx]):
    ax.text(imp + 0.001, bar.get_y() + bar.get_height()/2,
            f'{imp:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

### Exercise 5.2 — Model Evaluation Dashboard
Create a 2×2 dashboard that shows:
1. **Learning curves** — train vs val loss over 50 epochs
2. **Confusion matrix** — for the logistic regression model above
3. **Precision-Recall curve** — use `sklearn.metrics.precision_recall_curve`
4. **Prediction probability histogram** — separate histograms for true positives and true negatives

Hint: For plot 4, split `y_scores` by `y_test` values.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
# YOUR CODE HERE


In [ ]:
# SOLUTION
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Learning curves
ep = np.arange(1, 51)
tl = 1.8 * np.exp(-0.07*ep) + 0.1 + np.random.randn(50)*0.01
vl = 1.8 * np.exp(-0.05*ep) + 0.3 + np.random.randn(50)*0.02
axes[0,0].plot(ep, tl, label='Train', color='steelblue')
axes[0,0].plot(ep, vl, label='Val', color='coral')
axes[0,0].set_title('Learning Curves'); axes[0,0].legend()

# 2. Confusion matrix
y_pred_labels = (y_scores >= 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,1])
axes[0,1].set_xlabel('Predicted'); axes[0,1].set_ylabel('True')
axes[0,1].set_title('Confusion Matrix')

# 3. Precision-Recall
precision, recall, _ = precision_recall_curve(y_test, y_scores)
ap = average_precision_score(y_test, y_scores)
axes[1,0].plot(recall, precision, color='purple', lw=2)
axes[1,0].fill_between(recall, precision, alpha=0.1, color='purple')
axes[1,0].set_xlabel('Recall'); axes[1,0].set_ylabel('Precision')
axes[1,0].set_title(f'Precision-Recall Curve (AP={ap:.3f})')

# 4. Score distribution
scores_pos = y_scores[y_test == 1]
scores_neg = y_scores[y_test == 0]
axes[1,1].hist(scores_neg, bins=20, alpha=0.6, color='steelblue', label='True Negative')
axes[1,1].hist(scores_pos, bins=20, alpha=0.6, color='coral', label='True Positive')
axes[1,1].axvline(x=0.5, color='black', linestyle='--', label='Threshold')
axes[1,1].set_xlabel('Predicted Probability')
axes[1,1].set_title('Score Distribution by Class')
axes[1,1].legend()

plt.suptitle('Model Evaluation Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Challenge: Pair Plot & Outlier Detection Visualization

1. Create a **pair plot** (scatter matrix) of the numeric features in `df`, colored by `default` status
2. Create a **Z-score outlier plot**: standardize each numeric feature, flag points with |z| > 2.5, and show a scatter plot with outliers highlighted in red
3. Add a summary box in the plot corner showing outlier count per feature

In [ ]:
# YOUR CODE HERE


In [ ]:
# SOLUTION

# 1. Pair plot
plot_df = df[['age', 'income', 'credit_score', 'years_employed', 'default']].copy()
plot_df['default'] = plot_df['default'].map({0: 'No Default', 1: 'Default'})

g = sns.pairplot(plot_df, hue='default', diag_kind='kde', 
                  plot_kws={'alpha': 0.5}, height=2)
g.fig.suptitle('Pair Plot by Default Status', y=1.02, fontsize=14)
plt.show()

# 2 & 3. Outlier visualization
numeric_features = ['age', 'income', 'credit_score', 'years_employed']
z_scores = (df[numeric_features] - df[numeric_features].mean()) / df[numeric_features].std()
outlier_mask = (z_scores.abs() > 2.5).any(axis=1)

outlier_counts = (z_scores.abs() > 2.5).sum()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df.loc[~outlier_mask, 'income'], df.loc[~outlier_mask, 'credit_score'],
           alpha=0.5, c='steelblue', label='Normal', s=30)
ax.scatter(df.loc[outlier_mask, 'income'], df.loc[outlier_mask, 'credit_score'],
           alpha=0.8, c='red', label='Outlier (|z|>2.5)', s=60, marker='x', linewidths=2)

summary = '\n'.join([f"{f}: {c}" for f, c in outlier_counts.items()])
ax.text(0.02, 0.98, f"Outliers per feature:\n{summary}",
        transform=ax.transAxes, va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.set_xlabel('Income'); ax.set_ylabel('Credit Score')
ax.set_title(f'Outlier Detection (total: {outlier_mask.sum()} outliers)')
ax.legend()
plt.tight_layout()
plt.show()